In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt

# Step 1.1: Configure global publication-grade plotting parameters
scv.settings.set_figure_params('scanpy', dpi_save=300, frameon=False)
plt.rcParams['pdf.fonttype'] = 42      # Preserves editable TrueType paths for Illustrator/Acrobat
plt.rcParams['svg.fonttype'] = 'none'  # Retains texts as font metrics rather than path vectors

# Step 1.2: Path initialization
file_path = "/mnt/d/.Integrated_Final.h5ad"
save_dir = "/mnt/d/.all_scVelo_plt"
os.makedirs(save_dir, exist_ok=True)

print("Loading integrated transcriptomic velocity object matrix...")
adata = sc.read_h5ad(file_path)
print(f"Dataset successfully synchronized. Dimension metrics: {adata.shape}")

# Step 1.3: Apply lineage category mapping adjustments
print(f"Current structural annotations: {list(adata.obs['cell_type'].cat.categories)}")

new_names = {
    "ZIKV-Infected (Type 1)": "ZIKV-Infected RG",
    "ZIKV-Infected (Type 2)": "ZIKV-Infected NPCs",
    "ER-Stress Cells (ATF3+)": "Terminal Stress Cells",
    "ER-Stress Cells (CHOP+)": "Stress-response Progenitors",
    "Ribosomal-active Cells": "Active-state RGs",
}

# Safely update operational cluster labels across categorical levels
adata.obs['cell_type'] = adata.obs['cell_type'].cat.rename_categories(new_names)
print(f"Updated unified categories mapped: {list(adata.obs['cell_type'].cat.categories)}")

# Step 1.4: Multi-layer numerical sanitization
print("Executing matrix structural data cleanup for vector rendering compatibility...")

# Resolve infinite boundaries or undefined NaN states inside low-dimensional spatial matrices
if 'X_umap' in adata.obsm:
    adata.obsm['X_umap'] = np.nan_to_num(adata.obsm['X_umap'], nan=0.0, posinf=0.0, neginf=0.0)

# Clean all generated latent calculation layers related to structural dynamics
layers_to_clean = [l for l in adata.layers.keys() if 'velocity' in l]
for layer in layers_to_clean:
    adata.layers[layer] = np.nan_to_num(adata.layers[layer], nan=0.0, posinf=0.0, neginf=0.0)

# Step 1.5: High-resolution stream vector map rendering
print("Rendering global single-cell stream line embeddings...")
scv.pl.velocity_embedding_stream(
    adata, 
    basis='umap', 
    color='cell_type', 
    title='RNA Velocity of ZIKV-Infected Brain Organoids',
    legend_loc='right margin',
    legend_fontsize=9,
    size=50, 
    alpha=0.3,
    arrow_size=1.5,
    arrow_color='black',
    linewidth=1,
    min_mass=3,   # Filters out background stochastic noise artifacts
    show=False 
)

# Step 1.6: True-Type vector path serialization
save_path_svg = os.path.join(save_dir, "Figure1_scVelo_Streamline_Final.svg")

try:
    plt.savefig(save_path_svg, bbox_inches='tight', format='svg', transparent=True)
    print(f"[SUCCESS] Vector stream file committed to storage coordinate: {save_path_svg}")
except Exception as e:
    print(f"[WARNING] Native vector engine encountered exceptions: {e}. Executing PNG fallback matrix...")
    fallback_path = save_path_svg.replace(".svg", ".png")
    plt.savefig(fallback_path, bbox_inches='tight', dpi=300)
    print(f"[SUCCESS] Fallback raster chart written to: {fallback_path}")

plt.close()